# Xay dung Lap chi muc ngu nghia tiem an

In [103]:
import os
import nltk
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from whoosh.index import create_in
from whoosh.fields import *
from whoosh.analysis import StandardAnalyzer
from whoosh import qparser
from whoosh import scoring
import whoosh.index as index

import pytrec_eval
import math


nltk.download('punkt_tab')
nltk.download('stopwords')
stoplist = stopwords.words("english")
stoplist.append('oh')
puncts = ['.', ',', ':', '`', '"', "'", '!', '?', "``", "''"]
ps = PorterStemmer()
import shutil

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [104]:
def preprocess(tok, stemmer=ps, punctlist=puncts, stopwords=stoplist):
  tok = tok.lower()
  if tok.isdigit():
    return None
  if tok.isnumeric():
    return None
  if tok in punctlist:
    return None
  if tok in stopwords:
    return None
  return stemmer.stem(tok)

import shutil
import os
from whoosh.analysis import RegexTokenizer, LowercaseFilter, StopFilter, StemFilter

my_analyzer = (
    RegexTokenizer()
    | LowercaseFilter()
    | StopFilter(stoplist=stoplist)
    | StemFilter()
)

def indexing(src, idx="ind"):
    if os.path.exists(idx):
        shutil.rmtree(idx)

    os.mkdir(idx)

    schema = Schema(
        docid=ID(stored=True, unique=True),
        content=TEXT(stored=True, analyzer=StandardAnalyzer())
    )

    ix = create_in(idx, schema)
    writer = ix.writer()

    total_terms = 0          # tổng số token
    vocab = set()            # từ vựng (unique terms)

    if src[-1] != '/':
        src += '/'

    for f in os.listdir(src):
        with open(src + f, encoding="cp1252") as r:
            terms = []
            for s in r:
                for sent in sent_tokenize(s.strip()):
                    for tok in word_tokenize(sent):
                        tok = preprocess(tok)
                        if tok is not None:
                            terms.append(tok)
                            total_terms += 1
                            vocab.add(tok)

        writer.add_document(
            docid=f.split(".")[0],
            content=" ".join(terms)
        )

    writer.commit()

    print("Total terms (tokens):", total_terms)
    print("Vocabulary size (unique terms):", len(vocab))

In [105]:
indexing("../Cranfield/Cranfield", "ind")

Total terms (tokens): 131697
Vocabulary size (unique terms): 5373


In [106]:
import os
import shutil
import stat

def force_delete_directory(path):
    def onerror(func, path, exc_info):
        os.chmod(path, stat.S_IWRITE)
        func(path)

    if os.path.exists(path):
        shutil.rmtree(path, onerror=onerror)
        print(f"Đã xóa thư mục: {path}")

# force_delete_directory('index')

In [107]:
def readGroundTruth(src):
  if src[-1] != '/':
    src += '/'

  GT = {}
  for f in os.listdir(src):
    r = open(src + f)
    rel = {}
    for s in r:
      s = s.strip()
      sp = s.split("\t")
      if len(sp) < 2:
        continue
      did = sp[0].split(" ")[1]
      rel[did] = int(sp[1])
    GT[f.split(".")[0]] = rel
    r.close()
  return GT

In [108]:
GroundTruth = readGroundTruth("../Cranfield/RES")
print(GroundTruth)

{'1': {'184': 2, '29': 2, '31': 2, '12': 3, '51': 3, '102': 3, '13': 4, '14': 4, '15': 4, '57': 2, '378': 2, '859': 2, '185': 3, '30': 3, '37': 3, '52': 4, '142': 4, '195': 4, '875': 2, '56': 3, '66': 3, '95': 3, '462': 4, '497': 3, '858': 3, '876': 3, '879': 3, '880': 3, '486': -1}, '10': {'259': 2, '405': 2, '302': 3, '436': 3, '437': 3, '438': 3, '998': 3, '1011': 3, '493': -1}, '100': {'821': 2, '822': 2, '824': 2, '820': 3, '823': 3, '825': 3, '1122': 2, '1051': 3, '1121': 3, '760': -1}, '101': {'817': 2, '818': 2, '819': 2, '820': 3, '825': 3, '824': 2, '760': -1}, '102': {'728': 1, '913': 2, '910': 3, '911': 4, '729': -1}, '103': {'826': 3, '828': 3, '761': -1}, '104': {'833': 3, '834': 3, '835': 3, '836': 3, '837': 3, '762': -1}, '105': {'848': 3, '844': 4, '845': 4, '846': 4, '847': 4, '764': -1}, '106': {'847': 2, '846': 3, '849': 3, '844': 4, '845': 4, '764': -1}, '107': {'725': 2, '728': 2, '729': 3, '911': 3, '720': 4, '75': 4, '909': 4, '640': -1}, '108': {'724': 2, '726'

In [109]:
def readQuery(src):
  qry = {}
  r = open(src)
  for s in r:
    s = s.strip()
    ps = s.split("\t")
    terms = []
    for sent in sent_tokenize(ps[1]):
      for tok in word_tokenize(sent):
        tok = preprocess(tok)
        if tok != None:
          terms.append(tok)
    qry[ps[0]] = " ".join(terms)
  return qry

In [110]:
Queries = readQuery("../Cranfield/query.txt")
print(Queries)

{'1': 'similar law must obey construct aeroelast model heat high speed aircraft', '2': 'structur aeroelast problem associ flight high speed aircraft', '3': 'problem heat conduct composit slab solv far', '4': 'criterion develop show empir valid flow solut chemic react ga mixtur base simplifi assumpt instantan local chemic equilibrium', '5': 'chemic kinet system applic hyperson aerodynam problem', '6': 'theoret experiment guid turbul couett flow behaviour', '7': 'possibl relat avail pressur distribut ogiv forebodi zero angl attack lower surfac pressur equival ogiv forebodi angl attack', '8': 'method -dash exact approxim -dash present avail predict bodi pressur angl attack', '9': 'paper intern /slip flow/ heat transfer studi', '10': 'real-ga transport properti air avail wide rang enthalpi densiti', '11': 'possibl find analyt similar solut strong blast wave problem newtonian approxim', '12': 'aerodynam perform channel flow ground effect machin calcul', '13': 'basic mechan transon aileron b

In [111]:
from gensim import corpora, models, similarities

def readDocuments(src):
    docs = {}
    if src[-1] != '/':
        src += '/'
    for f in os.listdir(src):
        r = open(src + f, encoding="cp1252")
        terms = []
        for s in r:
            for sent in sent_tokenize(s.strip()):
                for tok in word_tokenize(sent):
                    tok = preprocess(tok)
                    if tok is not None:
                        terms.append(tok)
        r.close()
        docs[f.split(".")[0]] = terms
    return docs

documents = readDocuments("../Cranfield/Cranfield")
print(len(documents))


1400


In [112]:
dictionary = corpora.Dictionary(documents.values())
corpus = [dictionary.doc2bow(doc) for doc in documents.values()]
doc_ids = list(documents.keys())

In [150]:
tfidf = models.TfidfModel(corpus)
corpus_tfidf = tfidf[corpus]

In [151]:
NUM_TOPICS = 200

lsi = models.LsiModel(
    corpus_tfidf,
    id2word=dictionary,
    num_topics=NUM_TOPICS
)

corpus_lsi = lsi[corpus_tfidf]


In [152]:
index_lsi = similarities.MatrixSimilarity(
    corpus_lsi,
    num_features=NUM_TOPICS
)

In [153]:
def processQueries(qry, dictionary, tfidf, lsi, index_lsi, doc_ids, topk=None):
    RunResults = {}

    for qid, qtext in qry.items():
        q_tokens = qtext.split()
        q_bow = dictionary.doc2bow(q_tokens)
        q_tfidf = tfidf[q_bow]
        q_lsi = lsi[q_tfidf]

        sims = index_lsi[q_lsi]  # cosine similarity
        ranked = sorted(enumerate(sims), key=lambda x: -x[1])

        RunResults[qid] = {}
        if topk is None:
            topk = len(ranked)
        for rank, (doc_idx, score) in enumerate(ranked[:topk]):
            if score > 0:
                RunResults[qid][doc_ids[doc_idx]] = float(score)

    return RunResults


In [154]:
Queries = readQuery("../Cranfield/query.txt")
RunResults = processQueries(
    Queries,
    dictionary,
    tfidf,
    lsi,
    index_lsi,
    doc_ids
)


In [155]:
import pytrec_eval

def print_best_worst_queries(
    RunResults,
    Queries,
    GroundTruth,
    top_k=5,
    retrieved_k=10
):
    evaluator = pytrec_eval.RelevanceEvaluator(GroundTruth, {"map"})
    results = evaluator.evaluate(RunResults)

    # Lọc query hợp lệ (MAP != NaN)
    valid = [
        (qid, res["map"])
        for qid, res in results.items()
        if not math.isnan(res["map"])
    ]

    # Sort theo MAP
    valid_sorted = sorted(valid, key=lambda x: x[1], reverse=True)

    best = valid_sorted[:top_k]
    worst = valid_sorted[-top_k:]

    def print_block(title, items):
        print("\n" + "=" * 60)
        print(title)
        print("=" * 60)

        for qid, map_score in items:
            print(f"\nQuery ID : {qid}")
            print(f"Query    : {Queries[qid]}")
            print(f"MAP      : {map_score:.4f}")

            # Relevant docs
            rel_docs = [
                docid for docid, rel in GroundTruth[qid].items()
                if rel > 0
            ]
            print(f"Relevant docs ({len(rel_docs)}): {rel_docs[:retrieved_k]}")

            # Retrieved docs
            retrieved = list(RunResults[qid].keys())[:retrieved_k]
            print(f"Top retrieved docs: {retrieved}")

    print_block("🔥 TOP QUERIES (Highest MAP)", best)
    print_block("❄️ WORST QUERIES (Lowest MAP)", worst)


In [156]:
import pytrec_eval

# GroundTruth: dict {qid: {docid: relevance}}
# RunResults: dict {qid: {docid: score}}

# Chọn các metric cần tính
evaluator = pytrec_eval.RelevanceEvaluator(
    GroundTruth,
    {
        "map",       # Mean Average Precision
        "ndcg",
        "infAP",     # Inferred MAP
        "11pt_avg",
        "P_5", "P_10", "P_20",  # Precision@k
        "recall_5", "recall_10", "recall_20"  # Recall@k
    }
)

# Tính kết quả
results = evaluator.evaluate(RunResults)

# In kết quả từng query
for qid, metrics in results.items():
    print(f"Query {qid}")
    for m, v in metrics.items():
        print(f"  {m:10s}: {v:.4f}")
    print("-"*30)


Query 1
  map       : 0.2687
  P_5       : 0.6000
  P_10      : 0.6000
  P_20      : 0.4500
  recall_5  : 0.1071
  recall_10 : 0.2143
  recall_20 : 0.3214
  infAP     : 0.3251
  11pt_avg  : 0.3018
  ndcg      : 0.6022
------------------------------
Query 2
  map       : 0.1926
  P_5       : 0.6000
  P_10      : 0.3000
  P_20      : 0.3000
  recall_5  : 0.1250
  recall_10 : 0.1250
  recall_20 : 0.2500
  infAP     : 0.1955
  11pt_avg  : 0.2107
  ndcg      : 0.4878
------------------------------
Query 3
  map       : 0.4487
  P_5       : 0.4000
  P_10      : 0.5000
  P_20      : 0.3000
  recall_5  : 0.3333
  recall_10 : 0.8333
  recall_20 : 1.0000
  infAP     : 0.5918
  11pt_avg  : 0.5227
  ndcg      : 0.6431
------------------------------
Query 4
  map       : 0.7500
  P_5       : 0.4000
  P_10      : 0.2000
  P_20      : 0.1000
  recall_5  : 1.0000
  recall_10 : 1.0000
  recall_20 : 1.0000
  infAP     : 0.8750
  11pt_avg  : 0.7727
  ndcg      : 0.8772
------------------------------
Quer

In [158]:
# Tính trung bình (macro average) cho tất cả query
avg_metrics = {}
for m in next(iter(results.values())).keys():
    avg_metrics[m] = sum(q[m] for q in results.values()) / len(results)

# Tính macro F1 từ Precision và Recall trung bình
cutoffs = [5, 10, 20]

# Tính macro F1 cho các cutoff
for k in cutoffs:
    p_key = f"P_{k}"
    r_key = f"recall_{k}"
    f1_key = f"F1_{k}"
    
    if p_key in avg_metrics and r_key in avg_metrics:
        p = avg_metrics[p_key]
        r = avg_metrics[r_key]
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        avg_metrics[f1_key] = f1

# In ra
print("Average over all queries:")
for m, v in avg_metrics.items():
    print(f"  {m:10s}: {v:.4f}")


Average over all queries:
  map       : 0.3003
  P_5       : 0.2969
  P_10      : 0.2453
  P_20      : 0.1682
  recall_5  : 0.2635
  recall_10 : 0.4096
  recall_20 : 0.5339
  infAP     : 0.3506
  11pt_avg  : 0.3231
  ndcg      : 0.5058
  F1_5      : 0.2792
  F1_10     : 0.3069
  F1_20     : 0.2558


In [20]:
print(pytrec_eval.supported_measures)
eval = pytrec_eval.RelevanceEvaluator(GroundTruth, ["infAP", "11pt_avg"])

{'ndcg_rel', 'Rndcg', 'iprec_at_recall', 'ndcg_cut', 'utility', 'gm_bpref', 'set_relative_P', 'gm_map', 'recip_rank', 'num_rel', 'Rprec_mult', 'G', 'ndcg', 'success', 'relstring', 'relative_P', 'num_rel_ret', 'num_q', 'binG', 'num_nonrel_judged_ret', 'bpref', 'infAP', 'map_cut', '11pt_avg', 'num_ret', 'map', 'set_P', 'set_map', 'set_recall', 'recall', 'Rprec', 'set_F', 'P', 'runid'}


In [22]:
print_best_worst_queries(RunResults, Queries, GroundTruth)


🔥 TOP QUERIES (Highest MAP)

Query ID : 88
Query    : satellit orbit contract action air drag atmospher scale height vari altitud
MAP      : 0.9762
Relevant docs (6): ['613', '614', '615', '616', '617', '618']
Top retrieved docs: ['617', '615', '614', '613', '616', '548', '618', '622', '619', '510']

Query ID : 185
Query    : experiment studi panel flutter
MAP      : 0.8538
Relevant docs (9): ['858', '859', '857', '1008', '856', '15', '285', '894', '766']
Top retrieved docs: ['856', '1008', '766', '858', '859', '857', '391', '948', '864', '658']

Query ID : 41
Query    : anyon investig develop simpl model vortex wake behind cruciform wing
MAP      : 0.8333
Relevant docs (3): ['288', '289', '433']
Top retrieved docs: ['289', '433', '1152', '927', '1141', '288', '549', '1277', '126', '154']

Query ID : 154
Query    : iter method solv linear ellipt differ equat rapidli converg
MAP      : 0.8333
Relevant docs (2): ['1087', '1088']
Top retrieved docs: ['1088', '832', '1087', '1054', '1055'

## Lemma

In [39]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [40]:
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN


In [61]:
import unicodedata
from nltk import pos_tag

def preprocess_tokens(tokens, punctlist, stopwords):
    """
    tokens: list[str] đã được tokenize
    return: list[str] tokens sạch để indexing / querying
    """

    # Unicode normalize + lowercase
    tokens = [
        unicodedata.normalize("NFKC", tok.lower())
        for tok in tokens
    ]

    # POS tagging theo batch
    tagged = pos_tag(tokens)

    results = []

    for tok, pos in tagged:
        # loại số
        if tok.isdigit():
            continue

        # loại punctuation
        if tok in punctlist:
            continue

        # loại stopwords
        if tok in stopwords:
            continue

        # loại token quá ngắn
        if len(tok) <= 1:
            continue

        wn_pos = get_wordnet_pos(pos)
        lemma = lemmatizer.lemmatize(tok, wn_pos)

        results.append(lemma)

    return results

In [42]:
from whoosh.analysis import Filter
from nltk.stem import WordNetLemmatizer

class LemmaFilter(Filter):
    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()

    def __call__(self, tokens):
        for t in tokens:
            t.text = self.lemmatizer.lemmatize(t.text)
            yield t

In [62]:
import shutil
import os
from whoosh.analysis import RegexTokenizer, LowercaseFilter, StopFilter, StemFilter

lemma_analyzer = (
    RegexTokenizer()
    | LowercaseFilter()
    | StopFilter(stoplist=stoplist)
    | LemmaFilter()
)

def indexing(src, idx="ind"):
    if os.path.exists(idx):
        shutil.rmtree(idx)

    os.mkdir(idx)

    schema = Schema(
        docid=ID(stored=True, unique=True),
        content=TEXT(stored=True, analyzer=StandardAnalyzer())
    )

    ix = create_in(idx, schema)
    writer = ix.writer()

    total_terms = 0          # tổng số token
    vocab = set()            # từ vựng (unique terms)

    if src[-1] != '/':
        src += '/'

    for f in os.listdir(src):
        with open(src + f, encoding="cp1252") as r:
            terms = []
            for s in r:
                for sent in sent_tokenize(s.strip()):
                    for tok in word_tokenize(sent):
                        toks = preprocess_tokens([tok], punctlist=puncts, stopwords=stoplist)
                        for tok in toks:
                            terms.append(tok)
                            total_terms += 1
                            vocab.add(tok)

        writer.add_document(
            docid=f.split(".")[0],
            content=" ".join(terms)
        )

    writer.commit()

    print("Total terms (tokens):", total_terms)
    print("Vocabulary size (unique terms):", len(vocab))


In [63]:
indexing("../Cranfield/Cranfield", "ind")

Total terms (tokens): 129763
Vocabulary size (unique terms): 6370


In [64]:
GroundTruth = readGroundTruth("../Cranfield/RES")
print(GroundTruth)

{'1': {'184': 2, '29': 2, '31': 2, '12': 3, '51': 3, '102': 3, '13': 4, '14': 4, '15': 4, '57': 2, '378': 2, '859': 2, '185': 3, '30': 3, '37': 3, '52': 4, '142': 4, '195': 4, '875': 2, '56': 3, '66': 3, '95': 3, '462': 4, '497': 3, '858': 3, '876': 3, '879': 3, '880': 3, '486': -1}, '10': {'259': 2, '405': 2, '302': 3, '436': 3, '437': 3, '438': 3, '998': 3, '1011': 3, '493': -1}, '100': {'821': 2, '822': 2, '824': 2, '820': 3, '823': 3, '825': 3, '1122': 2, '1051': 3, '1121': 3, '760': -1}, '101': {'817': 2, '818': 2, '819': 2, '820': 3, '825': 3, '824': 2, '760': -1}, '102': {'728': 1, '913': 2, '910': 3, '911': 4, '729': -1}, '103': {'826': 3, '828': 3, '761': -1}, '104': {'833': 3, '834': 3, '835': 3, '836': 3, '837': 3, '762': -1}, '105': {'848': 3, '844': 4, '845': 4, '846': 4, '847': 4, '764': -1}, '106': {'847': 2, '846': 3, '849': 3, '844': 4, '845': 4, '764': -1}, '107': {'725': 2, '728': 2, '729': 3, '911': 3, '720': 4, '75': 4, '909': 4, '640': -1}, '108': {'724': 2, '726'

In [72]:
def readQuery_lemma(src):
  qry = {}
  r = open(src)
  for s in r:
    s = s.strip()
    ps = s.split("\t")
    terms = []
    for sent in sent_tokenize(ps[1]):
      for tok in word_tokenize(sent):
        toks = preprocess_tokens([tok], punctlist=puncts, stopwords=stoplist)
        for tok in toks:
          terms.append(tok)
    qry[ps[0]] = " ".join(terms)
  return qry

In [73]:
Queries = readQuery_lemma("../Cranfield/query.txt")
print(Queries)

{'1': 'similarity law must obeyed construct aeroelastic model heat high speed aircraft', '2': 'structural aeroelastic problem associate flight high speed aircraft', '3': 'problem heat conduction composite slab solve far', '4': 'criterion developed show empirically validity flow solution chemically react gas mixture base simplify assumption instantaneous local chemical equilibrium', '5': 'chemical kinetic system applicable hypersonic aerodynamic problem', '6': 'theoretical experimental guide turbulent couette flow behaviour', '7': 'possible relate available pressure distribution ogive forebody zero angle attack low surface pressure equivalent ogive forebody angle attack', '8': 'method -dash exact approximate -dash presently available predict body pressure angle attack', '9': 'paper internal /slip flow/ heat transfer study', '10': 'real-gas transport property air available wide range enthalpy density', '11': 'possible find analytical similar solution strong blast wave problem newtonian a

In [77]:
from gensim import corpora, models, similarities

def readDocuments_lemma(src):
    docs = {}
    if src[-1] != '/':
        src += '/'
    for f in os.listdir(src):
        r = open(src + f, encoding="cp1252")
        terms = []
        for s in r:
            for sent in sent_tokenize(s.strip()):
                for tok in word_tokenize(sent):
                    toks = preprocess_tokens([tok], punctlist=puncts, stopwords=stoplist)
                    for tok in toks:
                        terms.append(tok)
        r.close()
        docs[f.split(".")[0]] = terms
    return docs

In [78]:
documents = readDocuments_lemma("../Cranfield/Cranfield")
print(len(documents))

1400


In [79]:
dictionary = corpora.Dictionary(documents.values())
corpus = [dictionary.doc2bow(doc) for doc in documents.values()]
doc_ids = list(documents.keys())

In [80]:
tfidf = models.TfidfModel(corpus)
corpus_tfidf = tfidf[corpus]


In [81]:
NUM_TOPICS = 200

lsi = models.LsiModel(
    corpus_tfidf,
    id2word=dictionary,
    num_topics=NUM_TOPICS
)

corpus_lsi = lsi[corpus_tfidf]


In [ ]:
index_lsi = similarities.MatrixSimilarity(
    corpus_lsi,
    num_features=NUM_TOPICS
)

In [83]:
Queries = readQuery_lemma("../Cranfield/query.txt")
RunResults = processQueries(
    Queries,
    dictionary,
    tfidf,
    lsi,
    index_lsi,
    doc_ids
)


In [84]:
import pytrec_eval

# GroundTruth: dict {qid: {docid: relevance}}
# RunResults: dict {qid: {docid: score}}

# Chọn các metric cần tính
evaluator = pytrec_eval.RelevanceEvaluator(
    GroundTruth,
    {
        "map",       # Mean Average Precision
        "map_10",    # MAP@10
        "ndcg",
        "infAP",     # Inferred MAP
        "11pt_avg",
        "P_5", "P_10", "P_20",  # Precision@k
        "recall_5", "recall_10", "recall_20"  # Recall@k
    }
)

# Tính kết quả
results = evaluator.evaluate(RunResults)

# In kết quả từng query
for qid, metrics in results.items():
    print(f"Query {qid}")
    for m, v in metrics.items():
        print(f"  {m:10s}: {v:.4f}")
    print("-"*30)


Query 1
  map       : 0.2303
  P_5       : 0.6000
  P_10      : 0.5000
  P_20      : 0.3500
  recall_5  : 0.1071
  recall_10 : 0.1786
  recall_20 : 0.2500
  infAP     : 0.2782
  11pt_avg  : 0.2513
  ndcg      : 0.5620
------------------------------
Query 2
  map       : 0.1691
  P_5       : 0.4000
  P_10      : 0.3000
  P_20      : 0.2000
  recall_5  : 0.0833
  recall_10 : 0.1250
  recall_20 : 0.1667
  infAP     : 0.1719
  11pt_avg  : 0.1947
  ndcg      : 0.4547
------------------------------
Query 3
  map       : 0.5148
  P_5       : 0.6000
  P_10      : 0.6000
  P_20      : 0.3000
  recall_5  : 0.5000
  recall_10 : 1.0000
  recall_20 : 1.0000
  infAP     : 0.6736
  11pt_avg  : 0.6000
  ndcg      : 0.6727
------------------------------
Query 4
  map       : 1.0000
  P_5       : 0.4000
  P_10      : 0.2000
  P_20      : 0.1000
  recall_5  : 1.0000
  recall_10 : 1.0000
  recall_20 : 1.0000
  infAP     : 1.0000
  11pt_avg  : 1.0000
  ndcg      : 1.0000
------------------------------
Quer

In [85]:
# Tính trung bình (macro average) cho tất cả query
avg_metrics = {}
for m in next(iter(results.values())).keys():
    avg_metrics[m] = sum(q[m] for q in results.values()) / len(results)

# Tính macro F1 từ Precision và Recall trung bình
cutoffs = [5, 10, 20]

# Tính macro F1 cho các cutoff
for k in cutoffs:
    p_key = f"P_{k}"
    r_key = f"recall_{k}"
    f1_key = f"F1_{k}"
    
    if p_key in avg_metrics and r_key in avg_metrics:
        p = avg_metrics[p_key]
        r = avg_metrics[r_key]
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        avg_metrics[f1_key] = f1

# In ra
print("Average over all queries:")
for m, v in avg_metrics.items():
    print(f"  {m:10s}: {v:.4f}")

Average over all queries:
  map       : 0.2943
  P_5       : 0.2942
  P_10      : 0.2369
  P_20      : 0.1636
  recall_5  : 0.2560
  recall_10 : 0.3870
  recall_20 : 0.5282
  infAP     : 0.3422
  11pt_avg  : 0.3170
  ndcg      : 0.5022
  F1_5      : 0.2738
  F1_10     : 0.2939
  F1_20     : 0.2498


In [86]:
print_best_worst_queries(RunResults, Queries, GroundTruth)


🔥 TOP QUERIES (Highest MAP)

Query ID : 4
Query    : criterion developed show empirically validity flow solution chemically react gas mixture base simplify assumption instantaneous local chemical equilibrium
MAP      : 1.0000
Relevant docs (2): ['236', '166']
Top retrieved docs: ['236', '166', '167', '488', '317', '1286', '575', '656', '574', '259']

Query ID : 41
Query    : anyone investigate developed simple model vortex wake behind cruciform wing
MAP      : 1.0000
Relevant docs (3): ['288', '289', '433']
Top retrieved docs: ['289', '433', '288', '927', '1152', '1277', '1141', '549', '1196', '420']

Query ID : 154
Query    : iterative method solve linear elliptic difference equation rapidly convergent
MAP      : 1.0000
Relevant docs (2): ['1087', '1088']
Top retrieved docs: ['1088', '1087', '1054', '1055', '832', '1063', '111', '777', '1086', '454']

Query ID : 185
Query    : experimental study panel flutter
MAP      : 0.8379
Relevant docs (9): ['858', '859', '857', '1008', '856', '

### Indexing cuutom

In [88]:
import shutil
import os
from whoosh.analysis import RegexTokenizer, LowercaseFilter, StopFilter, StemFilter

lemma_analyzer = (
    RegexTokenizer()
    | LowercaseFilter()
    | StopFilter(stoplist=stoplist)
    | LemmaFilter()
)

def indexing_lemma(src, idx="ind"):
    if os.path.exists(idx):
        shutil.rmtree(idx)

    os.mkdir(idx)

    schema = Schema(
        docid=ID(stored=True, unique=True),
        content=TEXT(stored=True, analyzer=lemma_analyzer)
    )

    ix = create_in(idx, schema)
    writer = ix.writer()

    total_terms = 0          # tổng số token
    vocab = set()            # từ vựng (unique terms)

    if src[-1] != '/':
        src += '/'

    for f in os.listdir(src):
        with open(src + f, encoding="cp1252") as r:
            terms = []
            for s in r:
                for sent in sent_tokenize(s.strip()):
                    for tok in word_tokenize(sent):
                        toks = preprocess_tokens([tok], punctlist=puncts, stopwords=stoplist)
                        for tok in toks:
                            terms.append(tok)
                            total_terms += 1
                            vocab.add(tok)

        writer.add_document(
            docid=f.split(".")[0],
            content=" ".join(terms)
        )

    writer.commit()

    print("Total terms (tokens):", total_terms)
    print("Vocabulary size (unique terms):", len(vocab))


In [89]:
indexing_lemma("../Cranfield/Cranfield", "ind")

Total terms (tokens): 129763
Vocabulary size (unique terms): 6370


In [93]:
GroundTruth = readGroundTruth("../Cranfield/RES")
print(GroundTruth)

{'1': {'184': 2, '29': 2, '31': 2, '12': 3, '51': 3, '102': 3, '13': 4, '14': 4, '15': 4, '57': 2, '378': 2, '859': 2, '185': 3, '30': 3, '37': 3, '52': 4, '142': 4, '195': 4, '875': 2, '56': 3, '66': 3, '95': 3, '462': 4, '497': 3, '858': 3, '876': 3, '879': 3, '880': 3, '486': -1}, '10': {'259': 2, '405': 2, '302': 3, '436': 3, '437': 3, '438': 3, '998': 3, '1011': 3, '493': -1}, '100': {'821': 2, '822': 2, '824': 2, '820': 3, '823': 3, '825': 3, '1122': 2, '1051': 3, '1121': 3, '760': -1}, '101': {'817': 2, '818': 2, '819': 2, '820': 3, '825': 3, '824': 2, '760': -1}, '102': {'728': 1, '913': 2, '910': 3, '911': 4, '729': -1}, '103': {'826': 3, '828': 3, '761': -1}, '104': {'833': 3, '834': 3, '835': 3, '836': 3, '837': 3, '762': -1}, '105': {'848': 3, '844': 4, '845': 4, '846': 4, '847': 4, '764': -1}, '106': {'847': 2, '846': 3, '849': 3, '844': 4, '845': 4, '764': -1}, '107': {'725': 2, '728': 2, '729': 3, '911': 3, '720': 4, '75': 4, '909': 4, '640': -1}, '108': {'724': 2, '726'

In [94]:
documents = readDocuments_lemma("../Cranfield/Cranfield")
print(len(documents))

1400


In [95]:
dictionary = corpora.Dictionary(documents.values())
corpus = [dictionary.doc2bow(doc) for doc in documents.values()]
doc_ids = list(documents.keys())

In [96]:
tfidf = models.TfidfModel(corpus)
corpus_tfidf = tfidf[corpus]

In [97]:
NUM_TOPICS = 200

lsi = models.LsiModel(
    corpus_tfidf,
    id2word=dictionary,
    num_topics=NUM_TOPICS
)

corpus_lsi = lsi[corpus_tfidf]


In [98]:
index_lsi = similarities.MatrixSimilarity(
    corpus_lsi,
    num_features=NUM_TOPICS
)

In [99]:
Queries = readQuery_lemma("../Cranfield/query.txt")
RunResults = processQueries(
    Queries,
    dictionary,
    tfidf,
    lsi,
    index_lsi,
    doc_ids
)


In [100]:
import pytrec_eval

# GroundTruth: dict {qid: {docid: relevance}}
# RunResults: dict {qid: {docid: score}}

# Chọn các metric cần tính
evaluator = pytrec_eval.RelevanceEvaluator(
    GroundTruth,
    {
        "map",       # Mean Average Precision
        "map_10",    # MAP@10
        "ndcg",
        "infAP",     # Inferred MAP
        "11pt_avg",
        "P_5", "P_10", "P_20",  # Precision@k
        "recall_5", "recall_10", "recall_20"  # Recall@k
    }
)

# Tính kết quả
results = evaluator.evaluate(RunResults)

# In kết quả từng query
for qid, metrics in results.items():
    print(f"Query {qid}")
    for m, v in metrics.items():
        print(f"  {m:10s}: {v:.4f}")
    print("-"*30)


Query 1
  map       : 0.2318
  P_5       : 0.6000
  P_10      : 0.4000
  P_20      : 0.4000
  recall_5  : 0.1071
  recall_10 : 0.1429
  recall_20 : 0.2857
  infAP     : 0.2794
  11pt_avg  : 0.2452
  ndcg      : 0.5575
------------------------------
Query 2
  map       : 0.1679
  P_5       : 0.4000
  P_10      : 0.3000
  P_20      : 0.2000
  recall_5  : 0.0833
  recall_10 : 0.1250
  recall_20 : 0.1667
  infAP     : 0.1714
  11pt_avg  : 0.1928
  ndcg      : 0.4628
------------------------------
Query 3
  map       : 0.5148
  P_5       : 0.6000
  P_10      : 0.6000
  P_20      : 0.3000
  recall_5  : 0.5000
  recall_10 : 1.0000
  recall_20 : 1.0000
  infAP     : 0.6736
  11pt_avg  : 0.6000
  ndcg      : 0.6727
------------------------------
Query 4
  map       : 1.0000
  P_5       : 0.4000
  P_10      : 0.2000
  P_20      : 0.1000
  recall_5  : 1.0000
  recall_10 : 1.0000
  recall_20 : 1.0000
  infAP     : 1.0000
  11pt_avg  : 1.0000
  ndcg      : 1.0000
------------------------------
Quer

In [101]:
# Tính trung bình (macro average) cho tất cả query
avg_metrics = {}
for m in next(iter(results.values())).keys():
    avg_metrics[m] = sum(q[m] for q in results.values()) / len(results)

# Tính macro F1 từ Precision và Recall trung bình
cutoffs = [5, 10, 20]

# Tính macro F1 cho các cutoff
for k in cutoffs:
    p_key = f"P_{k}"
    r_key = f"recall_{k}"
    f1_key = f"F1_{k}"
    
    if p_key in avg_metrics and r_key in avg_metrics:
        p = avg_metrics[p_key]
        r = avg_metrics[r_key]
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        avg_metrics[f1_key] = f1

# In ra
print("Average over all queries:")
for m, v in avg_metrics.items():
    print(f"  {m:10s}: {v:.4f}")

Average over all queries:
  map       : 0.2948
  P_5       : 0.2933
  P_10      : 0.2360
  P_20      : 0.1649
  recall_5  : 0.2541
  recall_10 : 0.3895
  recall_20 : 0.5307
  infAP     : 0.3433
  11pt_avg  : 0.3168
  ndcg      : 0.5012
  F1_5      : 0.2723
  F1_10     : 0.2939
  F1_20     : 0.2516


In [102]:
print_best_worst_queries(RunResults, Queries, GroundTruth)


🔥 TOP QUERIES (Highest MAP)

Query ID : 4
Query    : criterion developed show empirically validity flow solution chemically react gas mixture base simplify assumption instantaneous local chemical equilibrium
MAP      : 1.0000
Relevant docs (2): ['236', '166']
Top retrieved docs: ['236', '166', '167', '488', '317', '1286', '1296', '575', '656', '259']

Query ID : 41
Query    : anyone investigate developed simple model vortex wake behind cruciform wing
MAP      : 1.0000
Relevant docs (3): ['288', '289', '433']
Top retrieved docs: ['289', '433', '288', '927', '1152', '1277', '1141', '549', '154', '420']

Query ID : 154
Query    : iterative method solve linear elliptic difference equation rapidly convergent
MAP      : 1.0000
Relevant docs (2): ['1087', '1088']
Top retrieved docs: ['1088', '1087', '1054', '1055', '832', '111', '1063', '1262', '777', '1086']

Query ID : 185
Query    : experimental study panel flutter
MAP      : 0.8546
Relevant docs (9): ['858', '859', '857', '1008', '856', 